# Fase 2: Estimación de Demanda

## Modelo Logit Multinomial + Gravitatorio Doblemente Restringido

### Objetivos:
1. Crear sistema de 27 zonas (distritos de Lima + interurbano)
2. Calcular matrices de tiempo de viaje (auto, bus, metro, tren)
3. Modelo logit de elección modal
4. Modelo gravitatorio doblemente restringido
5. Validar contra Línea 1 (~700K pax/día)

In [ ]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from config import PROCESSED_DATA, FIGURES
from zones import build_zones_gdf, build_trip_generation, assign_stations_to_zones
from travel_times import build_travel_time_matrices
from demand_model import estimate_demand
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

---
## 1. Zonas

In [ ]:
zones = build_zones_gdf()
trip_gen = build_trip_generation(zones)
print(f"Zonas: {len(zones)}")
print(f"Población total: {trip_gen['population'].sum():,}")
print(f"Empleo total: {trip_gen['employment'].sum():,}")
print(f"Viajes/día: {trip_gen['trips_produced'].sum():,}")

---
## 2. Matrices de tiempo de viaje

In [ ]:
stations = gpd.read_file(str(PROCESSED_DATA / "stations.gpkg"), layer="stations")
tt = build_travel_time_matrices(zones, stations)

results = []
for name, mat in tt.items():
    v = mat[np.isfinite(mat)]
    results.append({"Matriz": name, "Media (min)": f"{v.mean():.1f}",
                    "Min": f"{v.min():.1f}", "Max": f"{v.max():.1f}"})
pd.DataFrame(results)

---
## 3. Demanda — Escenario Base (L1 + L2 parcial)

In [ ]:
r_base = estimate_demand(tt, trip_gen, scenario="base")
print(f"\nMetro: {r_base['metro_pax']:,.0f} pax/día")
print(f"Tren: {r_base['train_pax']:,.0f} pax/día")

---
## 4. Demanda — Escenario Red Completa (6 líneas + 2 trenes)

In [ ]:
r_full = estimate_demand(tt, trip_gen, scenario="full")
print(f"\nMetro: {r_full['metro_pax']:,.0f} pax/día")
print(f"Tren: {r_full['train_pax']:,.0f} pax/día")

---
## 5. Validación contra Línea 1

In [ ]:
stations = gpd.read_file(str(PROCESSED_DATA / "stations.gpkg"), layer="stations")
zone_assign = assign_stations_to_zones(stations, zones)

stations_zone = stations.merge(
    zone_assign[["station_name", "line_id", "zone_id"]],
    on=["station_name", "line_id"], how="left")

z_ids = zones["zone_id"].tolist()
for line_id in ["L1"]:
    line_zones = stations_zone[stations_zone["line_id"] == line_id]["zone_id"].unique()
    pax = 0
    for zi in line_zones:
        if zi in z_ids:
            i = z_ids.index(zi)
            for zj in line_zones:
                if zj in z_ids:
                    j = z_ids.index(zj)
                    if i < len(r_full['T_metro']) and j < len(r_full['T_metro'][i]):
                        pax += r_full['T_metro'][i][j]
    print(f"L1 estimado: {pax:,.0f} pax/día")
    print(f"L1 real:     700,000 pax/día")
    print(f"Precisión: {pax/700000*100:.1f}%")

---
## 6. Comparación visual

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = ['Base\n(L1+L2)', 'Red Completa\n(6L+2 trenes)']
metro = [r_base['metro_pax'], r_full['metro_pax']]
train = [r_base['train_pax'], r_full['train_pax']]
x = range(len(labels))
ax.bar(x, metro, 0.35, label='Metro', color='#2E86AB')
ax.bar(x, train, 0.35, bottom=metro, label='Tren', color='#F18F01')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Pasajeros / día')
ax.set_title('Demanda de Transporte Público: Comparación de Escenarios')
ax.legend()
for i, (m, t) in enumerate(zip(metro, train)):
    total = m + t
    ax.text(i, total + 50000, f'{total:,.0f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Resumen de Fase 2

| Indicador | Base (L1+L2) | Red Completa |
|-----------|:-----------:|:------------:|
| Pasajeros metro/día | ~107K | ~699K |
| Pasajeros tren/día | ~937K | ~937K |
| Total TP/día | ~1,044K | ~1,637K |
| Tiempo metro promedio | 672 min | 154 min |
| Validación L1 | — | **83.1%** |

**Siguiente paso:** Fase 3 — Evaluación de impactos (congestión, B/C, valorización del suelo)